In [ ]:
# # =========================
# # Imports & Device Setup
# # =========================
# import os, glob, random
# import numpy as np
# from PIL import Image
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# import seaborn as sns

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader, TensorDataset
# from torchvision import transforms
# import timm

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# # Device
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device:", device)

# # Seed for reproducibility
# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(42)


# # =========================
# # Dataset Paths & Split
# # =========================
# DATASET_DIR = r"E:\ViT-based-Framework-for-Multi-Class-Classification-of-Mango-Leaf-Diseases\dataset"
# IMG_SIZE = 224

# # Classes
# CLASS_NAMES = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
# num_classes = len(CLASS_NAMES)
# print("Classes:", CLASS_NAMES)

# # Image paths and labels
# image_paths, labels = [], []
# for lbl in CLASS_NAMES:
#     folder = os.path.join(DATASET_DIR, lbl)
#     imgs = glob.glob(os.path.join(folder, '*.jpg')) + glob.glob(os.path.join(folder, '*.png'))
#     image_paths.extend(imgs)
#     labels.extend([lbl]*len(imgs))

# # Encode labels
# le = LabelEncoder()
# labels_encoded = le.fit_transform(labels)

# # 70:15:15 split
# X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
#     image_paths, labels_encoded, stratify=labels_encoded, test_size=0.3, random_state=42
# )
# X_val_paths, X_test_paths, y_val, y_test = train_test_split(
#     X_temp_paths, y_temp, stratify=y_temp, test_size=0.5, random_state=42
# )
# print(f"Train: {len(X_train_paths)}  Val: {len(X_val_paths)}  Test: {len(X_test_paths)}")


# # =========================
# # Image Transforms
# # =========================
# train_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(10),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
#     transforms.ToTensor(),
#     transforms.Normalize([0.5]*3, [0.5]*3)
# ])

# val_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize([0.5]*3, [0.5]*3)
# ])


# # =========================
# # Feature Extraction (Tiny Swin V2)
# # =========================
# model_name = 'swin_tiny_patch4_window7_224'
# feature_model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 for features
# feature_model.eval().to(device)

# @torch.no_grad()
# def extract_features(paths, transform):
#     feats = []
#     for path in tqdm(paths):
#         img = Image.open(path).convert('RGB')
#         x = transform(img).unsqueeze(0).to(device)
#         feat = feature_model(x)
#         feats.append(feat.squeeze(0).cpu().numpy())
#     return np.stack(feats)

# print("Extracting features...")
# X_train_feats = extract_features(X_train_paths, train_transform)
# X_val_feats   = extract_features(X_val_paths, val_transform)
# X_test_feats  = extract_features(X_test_paths, val_transform)

# print("Feature shapes:", X_train_feats.shape, X_val_feats.shape, X_test_feats.shape)


# # =========================
# # LoRA Linear Head
# # =========================
# class LoRALinear(nn.Module):
#     def __init__(self, in_features, out_features, r=16, alpha=32):
#         super().__init__()
#         self.r = r
#         self.alpha = alpha
#         self.scale = alpha / r
#         self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
#         self.bias = nn.Parameter(torch.zeros(out_features))
#         self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
#         self.lora_B = nn.Parameter(torch.randn(out_features, r) * 0.01)

#     def forward(self, x):
#         return F.linear(x, self.weight + (self.lora_B @ self.lora_A) * self.scale, self.bias)


# # Instantiate model
# in_features = X_train_feats.shape[1]
# model_head = LoRALinear(in_features, num_classes, r=16, alpha=32).to(device)


# # =========================
# # Dataloaders
# # =========================
# batch_size = 32

# train_loader = DataLoader(TensorDataset(torch.tensor(X_train_feats, dtype=torch.float32),
#                                         torch.tensor(y_train, dtype=torch.long)),
#                           batch_size=batch_size, shuffle=True)

# val_loader = DataLoader(TensorDataset(torch.tensor(X_val_feats, dtype=torch.float32),
#                                       torch.tensor(y_val, dtype=torch.long)),
#                         batch_size=batch_size, shuffle=False)

# test_loader = DataLoader(TensorDataset(torch.tensor(X_test_feats, dtype=torch.float32),
#                                        torch.tensor(y_test, dtype=torch.long)),
#                          batch_size=batch_size, shuffle=False)


# # =========================
# # Training Loop with Early Stopping
# # =========================
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model_head.parameters(), lr=1e-4, weight_decay=1e-4)
# EPOCHS = 50
# PATIENCE = 5
# best_val_loss = float("inf")
# wait = 0
# best_model_path = "best_model.pth"

# train_losses, val_losses = [], []

# for epoch in range(EPOCHS):
#     # Training
#     model_head.train()
#     total_loss = 0.0
#     for xb, yb in train_loader:
#         xb, yb = xb.to(device), yb.to(device)
#         optimizer.zero_grad()
#         logits = model_head(xb)
#         loss = criterion(logits, yb)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item() * xb.size(0)
#     epoch_loss = total_loss / len(train_loader.dataset)
#     train_losses.append(epoch_loss)

#     # Validation
#     model_head.eval()
#     val_loss = 0.0
#     with torch.no_grad():
#         for xb, yb in val_loader:
#             xb, yb = xb.to(device), yb.to(device)
#             logits = model_head(xb)
#             loss = criterion(logits, yb)
#             val_loss += loss.item() * xb.size(0)
#     val_loss /= len(val_loader.dataset)
#     val_losses.append(val_loss)

#     print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {epoch_loss:.4f}  Val Loss: {val_loss:.4f}")

#     # Early Stopping
#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         wait = 0
#         torch.save(model_head.state_dict(), best_model_path)
#     else:
#         wait += 1
#         if wait >= PATIENCE:
#             print(f"Early stopping at epoch {epoch+1}")
#             break

# # Plot losses
# plt.plot(train_losses, label="Train Loss")
# plt.plot(val_losses, label="Val Loss")
# plt.xlabel("Epochs")
# plt.ylabel("Loss")
# plt.title("Training vs Validation Loss")
# plt.legend()
# plt.show()


# # =========================
# # Evaluation
# # =========================
# model_head.load_state_dict(torch.load(best_model_path))
# model_head.eval()

# y_true, y_pred = [], []
# with torch.no_grad():
#     for xb, yb in test_loader:
#         xb = xb.to(device)
#         outputs = model_head(xb)
#         preds = torch.argmax(outputs, dim=1).cpu().numpy()
#         y_true.extend(yb.numpy())
#         y_pred.extend(preds)

# accuracy = accuracy_score(y_true, y_pred)
# precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")

# print("\nFinal Test Metrics:")
# print(f"Accuracy  : {accuracy:.4f}")
# print(f"Precision : {precision:.4f}")
# print(f"Recall    : {recall:.4f}")
# print(f"F1 Score  : {f1:.4f}")

# # Confusion Matrix
# cm = confusion_matrix(y_true, y_pred)
# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.title("Confusion Matrix")
# plt.show()


In [1]:
# ============================
# Baseline: Swin-Tiny + LoRA Head
# ============================

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ----------------------------
# Config
# ----------------------------
DATASET_DIR = "E:\ViT-based-Framework-for-Multi-Class-Classification-of-Mango-Leaf-Diseases\dataset"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
PATIENCE = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Dataset
# ----------------------------
class LeafDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = self.labels[idx]
        return img, label

# ----------------------------
# Data Prep
# ----------------------------
classes = sorted(os.listdir(DATASET_DIR))
all_images, all_labels = [], []

for label in classes:
    img_paths = glob.glob(os.path.join(DATASET_DIR, label, "*.jpg"))
    all_images.extend(img_paths)
    all_labels.extend([label] * len(img_paths))

# Encode labels
le = LabelEncoder()
all_labels_enc = le.fit_transform(all_labels)

# Train-val-test split (70-15-15)
X_train, X_temp, y_train, y_temp = train_test_split(
    all_images, all_labels_enc, test_size=0.30, stratify=all_labels_enc, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# ----------------------------
# Transforms (mild aug for baseline)
# ----------------------------
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_ds = LeafDataset(X_train, y_train, transform=train_transform)
val_ds = LeafDataset(X_val, y_val, transform=test_transform)
test_ds = LeafDataset(X_test, y_test, transform=test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

e:\ViT-based-Framework-for-Multi-Class-Classification-of-Mango-Leaf-Diseases\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# Feature Extractor (Swin-Tiny)
# ----------------------------
feature_extractor = timm.create_model(
    "swin_tiny_patch4_window7_224", pretrained=True, num_classes=0
).to(DEVICE)
feature_extractor.eval()  # frozen
for p in feature_extractor.parameters():
    p.requires_grad = False

# Get feature dimension
with torch.no_grad():
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    feat_dim = feature_extractor(dummy).shape[1]

In [3]:
# ----------------------------
# LoRA Linear Head
# ----------------------------
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4, alpha=8):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.scale = alpha / r

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.randn(out_features, r) * 0.01)

    def forward(self, x):
        return nn.functional.linear(
            x, self.weight + self.scale * (self.lora_B @ self.lora_A), self.bias
        )

model_head = LoRALinear(feat_dim, len(classes), r=4, alpha=8).to(DEVICE)

In [4]:
# ----------------------------
# Training Loop
# ----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_head.parameters(), lr=5e-5)

best_val_loss = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    # Train
    model_head.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            feats = feature_extractor(imgs)
        preds = model_head(feats)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    model_head.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            feats = feature_extractor(imgs)
            preds = model_head(feats)
            loss = criterion(preds, labels)
            val_loss += loss.item() * imgs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_head.state_dict(), "baseline_lora_head.pth")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping!")
            break

Epoch 1/20 | Train Loss: 1.9237 | Val Loss: 1.7586
Epoch 2/20 | Train Loss: 1.6086 | Val Loss: 1.4781
Epoch 3/20 | Train Loss: 1.3428 | Val Loss: 1.2437
Epoch 4/20 | Train Loss: 1.1258 | Val Loss: 1.0564
Epoch 5/20 | Train Loss: 0.9532 | Val Loss: 0.9083
Epoch 6/20 | Train Loss: 0.8164 | Val Loss: 0.7913
Epoch 7/20 | Train Loss: 0.7100 | Val Loss: 0.6975
Epoch 8/20 | Train Loss: 0.6240 | Val Loss: 0.6220
Epoch 9/20 | Train Loss: 0.5550 | Val Loss: 0.5596
Epoch 10/20 | Train Loss: 0.4976 | Val Loss: 0.5074
Epoch 11/20 | Train Loss: 0.4488 | Val Loss: 0.4635
Epoch 12/20 | Train Loss: 0.4078 | Val Loss: 0.4258
Epoch 13/20 | Train Loss: 0.3734 | Val Loss: 0.3929
Epoch 14/20 | Train Loss: 0.3442 | Val Loss: 0.3641
Epoch 15/20 | Train Loss: 0.3166 | Val Loss: 0.3390
Epoch 16/20 | Train Loss: 0.2940 | Val Loss: 0.3167
Epoch 17/20 | Train Loss: 0.2719 | Val Loss: 0.2969
Epoch 18/20 | Train Loss: 0.2548 | Val Loss: 0.2790
Epoch 19/20 | Train Loss: 0.2384 | Val Loss: 0.2628
Epoch 20/20 | Train L

In [ ]:
# ----------------------------
# Evaluation
# ----------------------------
model_head.load_state_dict(torch.load("baseline_lora_head.pth"))
model_head.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        feats = feature_extractor(imgs)
        preds = model_head(feats)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(torch.argmax(preds, dim=1).cpu().numpy())

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

print("\n=== Test Metrics (Baseline) ===")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")